In [ ]:
/**
 * DATABASE SCALING — Partitioning and Data Lifecycle
 * 1.  Why Partitioning? ................ When tables become too large
 * 2.  Partition vs Partitioning ........ One logical table, many physical slices
 * 3.  Partition By Time ................ The most common strategy
 * 4.  How Partitioning Helps ........... Reducing data scanned for queries
 * 5.  What Partitioning Doesn't Do ..... Not a magic speed booster
 * 6.  Partitioning vs Sharding ......... One DB vs many DBs
 * 7.  Data Lifecycle ................... Hot, Warm, and Cold data
 * 8.  How Partitioning Helps Lifecycle . Operate at partition level
 * 9.  Delete vs Anonymize .............. Complete removal vs identity masking
 * 10. What Decides Lifecycle Policy .... Business, legal, and operational factors
 * 11. Golden Rules ..................... Key principles to remember
 */

In [ ]:
/**
 * SAMPLE TABLE — orders (For all partitioning examples)
 *
 *   5 saal ki order history store ki gayi hai — billions of rows.
 *   Users mostly last 30 days ka data dekhte hain.
 *   Par DB poora 5 saal ka structure operate karta hai — even for recent queries.
 *
 * ┌──────────┬─────────┬─────────────────────┬──────────────┬──────────────┐
 * │ order_id │ user_id │ created_at          │ status       │ total_amount │
 * ├──────────┼─────────┼─────────────────────┼──────────────┼──────────────┤
 * │ 1        │ 101     │ 2026-01-15 10:30:00 │ completed    │ 99.99        │
 * │ 2        │ 102     │ 2026-01-20 14:22:00 │ pending      │ 49.99        │
 * │ 3        │ 101     │ 2026-02-05 09:15:00 │ completed    │ 149.99       │
 * │ ...      │ ...     │ ...                 │ ...          │ ...          │
 * │ 1M+      │ ...     │ 2021-01-01 00:00:00 │ completed    │ 29.99        │
 * └──────────┴─────────┴─────────────────────┴──────────────┴──────────────┘
 *
 * 5+ years of order history — billions of rows
 */

In [ ]:
/**
 * Part-1: WHY PARTITIONING?
 */

In [ ]:
/**
 * Jab Tables Bahut Badi Ho Jaayein
 * ──────────────────────────────────
 *
 *   Pehle humne indexes, replicas, sharding sab cover kiya.
 *   Ab ek naya question: data ek DB pe fit hota hai lekin table ITNI BADI hai
 *   ki queries slow ho rahi hain.
 *   PARTITIONING is problem solve karta hai.
 *
 * WE'VE ALREADY COVERED:
 *   → How indexes help reads
 *   → How replicas scale reads
 *   → How sharding helps when one writer is not enough
 *   → How schema changes must be handled carefully
 *
 * NOW A NEW QUESTION:
 *   "What if data still fits on one database server,
 *    but queries are becoming slower because the table
 *    has become TOO LARGE?"
 *
 *   That's where PARTITIONING comes in.
 *
 * REAL-WORLD SCENARIO:
 *   Orders table stores 5 years of order history.
 *   When users open the orders page:
 *   → Most check LAST 30 DAYS
 *   → Some check last 3-6 months
 *   → Few check 1+ year old orders
 *
 *   But the database still operates on a MASSIVE STRUCTURE
 *   holding 5 years of data — even when users only need recent.
 *
 *   Partitioning breaks one logical table into smaller physical slices.
 */

In [ ]:
/**
 * Part-2: PARTITION VS PARTITIONING
 */

In [ ]:
/**
 * Ek Logical Table, Many Physical Slices
 * ─────────────────────────────────────────
 *
 *   Partition = ek physical slice of a table.
 *   Partitioning = process of breaking one logical table into many such slices.
 *   App ke liye kuch nahi badlata — same SQL, same table name.
 *   Lekin andar se DB smaller pieces mein kaam karta hai.
 *
 * A PARTITION = one physical slice of a table.
 * PARTITIONING = the process of breaking one logical table
 *                into many such physical slices.
 *
 * BEFORE PARTITIONING:
 *   ┌──────────────────────────────────────────────────────────┐
 *   │                    ORDERS TABLE                          │
 *   │   (all 5 years of data in one giant structure)           │
 *   └──────────────────────────────────────────────────────────┘
 *
 * AFTER PARTITIONING (by month):
 *
 *   ┌──────────────┐ ┌──────────────┐ ┌──────────────┐
 *   │ Jan 2026     │ │ Feb 2026     │ │ Mar 2026     │  ...
 *   │ partition    │ │ partition    │ │ partition    │
 *   └──────────────┘ └──────────────┘ └──────────────┘
 *
 * THE USER EXPERIENCE:
 *   Application code doesn't change.
 *   Same SQL queries work the same way.
 *
 *   But internally, the database knows:
 *   → Jan data lives HERE
 *   → Feb data lives HERE
 *   → Mar data lives HERE
 *
 *   This allows the database to work with SMALLER PIECES
 *   rather than one massive structure.
 */

In [ ]:
/**
 * Part-3: PARTITION BY TIME
 */

In [ ]:
/**
 * Sabse Common Strategy — Date/Month Ke Hisaab Se
 * ──────────────────────────────────────────────────
 *
 *   Transactional tables ke liye time-based partitioning sabse common hai.
 *   Har month ya year ka alag partition banao.
 *   Lifecycle management bhi bahut easy ho jaata hai.
 *
 * MOST COMMON PARTITIONING STRATEGY for transactional tables:
 *   PARTITION BY DATE or MONTH
 *
 * EXAMPLE: Partition orders by month
 *
 *   Partition p2026_01: orders from January 2026
 *   Partition p2026_02: orders from February 2026
 *   Partition p2026_03: orders from March 2026
 *   Partition p2025_12: orders from December 2025
 *   ...
 *   Partition p2021_01: orders from January 2021
*/

/**
 * MYSQL EXAMPLE (Range Partitioning):
 */


CREATE TABLE orders (
    order_id     INT NOT NULL,
    user_id      INT NOT NULL,
    created_at   DATETIME NOT NULL,
    status       VARCHAR(20),
    total_amount DECIMAL(10,2)
)
PARTITION BY RANGE (YEAR(created_at) * 100 + MONTH(created_at))
(
    PARTITION p2025_01 VALUES LESS THAN (202502),
    PARTITION p2025_02 VALUES LESS THAN (202503),
    PARTITION p2025_03 VALUES LESS THAN (202504)
    -- ...
);


/**
 * POSTGRESQL EXAMPLE:
 */


CREATE TABLE orders (
    order_id     INT NOT NULL,
    user_id      INT NOT NULL,
    created_at   TIMESTAMP NOT NULL,
    status       VARCHAR(20),
    total_amount DECIMAL(10,2)
) PARTITION BY RANGE (created_at);

-- Individual partition tables:
CREATE TABLE orders_2025_01 PARTITION OF orders
FOR VALUES FROM ('2025-01-01') TO ('2025-02-01');

CREATE TABLE orders_2025_02 PARTITION OF orders
FOR VALUES FROM ('2025-02-01') TO ('2025-03-01');

In [ ]:
/**
 * Part-4: HOW PARTITIONING HELPS
 */

In [ ]:
/**
 * Partition Pruning — Sirf Relevant Data Scan Karo
 * ──────────────────────────────────────────────────
 *
 *   Partitioning ka main benefit hai PRUNING.
 *   DB ko pata hota hai kaun se partitions relevant hain — baaki IGNORE karo.
 *   5 saal ki jagah sirf 3 mahine ka data scan hota hai — MUCH FASTER!
 *
 * THE MAIN BENEFIT: PRUNING
 *
 *   Pruning means:
 *   The database knows which partitions to look at
 *   and which partitions to IGNORE.
 *
 * EXAMPLE QUERY: Orders from January-March 2026
 */


SELECT * FROM orders
WHERE created_at BETWEEN '2026-01-01' AND '2026-03-31';


/**
 * WITHOUT PARTITIONING:
 *   → Database scans the ENTIRE orders table
 *   → All 5 years of data
 *   → Millions or billions of rows
 *   → SLOW!
 *
 * WITH PARTITIONING (by month):
 *   → Database knows: only Jan, Feb, Mar 2026 partitions matter
 *   → Ignores all other partitions
 *   → Scans only 3 months of data instead of 5 years
 *   → MUCH FASTER!
 *
 * VISUAL:
 *
 *   Without Partitioning:
 *   ┌──────────────────────────────────────────────────────────┐
 *   │ ██████████████████████████████████████████████████████   │
 *   │              Scan ALL 5 years (slow)                     │
 *   └──────────────────────────────────────────────────────────┘
 *
 *   With Partitioning:
 *   ┌─────┐┌─────┐┌─────┐┌─────┐┌─────┐┌─────┐
 *   │ Jan ││ Feb ││ Mar ││ Apr ││ ... ││ Dec │
 *   │2026 ││2026 ││2026 ││2026 ││     ││2025 │
 *   └──█──┘└──█──┘└──█──┘└─────┘└─────┘└─────┘
 *      █      █      █
 *      └──────┴──────┴──► Scan ONLY these 3 partitions ✅
 */

In [ ]:
/**
 * Part-5: WHAT PARTITIONING DOESN'T DO
 */

In [ ]:
/**
 * Magic Speed Booster Nahi Hai — Warning!
 * ──────────────────────────────────────────
 *
 *   Ye sabse important WARNING hai partitioning ke baare mein.
 *   Partitioning tabhi help karta hai jab query mein partition key ho.
 *   user_id se search karo date-partitioned table pe — koi fayda nahi.
 *
 * PARTITIONING IS NOT A MAGIC SPEED BOOSTER FOR EVERY QUERY.
*/

/**
 * EXAMPLE WHERE PARTITIONING DOESN'T HELP:
 * → This query needs all orders for a specific user
 * → User may have orders across many years
 * → Database may need to scan ALL partitions
 * → Date partitioning doesn't help here at all!
 */


SELECT * FROM orders WHERE user_id = 101;


/**
 * PARTITIONING HELPS MOST WHEN:
 *   → Queries naturally include the partition key
 *   → Example: date ranges on a date-partitioned table
 *   → Example: region on a region-partitioned table
 *
 * PARTITIONING DOESN'T HELP WHEN:
 *   → Queries don't include the partition key
 *   → Example: user_id queries on date-partitioned table
 *   → Example: status queries on date-partitioned table
 *
 * THE KEY PRINCIPLE:
 *   Partitioning must MATCH your actual query patterns.
 *   Choose partition key based on HOW YOU QUERY the data.
 *
 *   If most queries are:
 *   → By date range  → partition by date
 *   → By region      → partition by region
 *   → By tenant      → partition by tenant_id
 *
 *   Partition key that doesn't match access patterns = wasted effort.
 */

In [ ]:
/**
 * Part-6: PARTITIONING VS SHARDING
 */

In [ ]:
/**
 * Ek DB vs Many DBs — Crystal Clear Difference
 * ───────────────────────────────────────────────
 *
 *   Ye frequently asked interview question hai.
 *   Partitioning = ek DB ke andar.
 *   Sharding = multiple DBs mein across.
 *   Dono alag problems solve karte hain.
 *
 * PARTITIONING:
 *   → Happens INSIDE one database instance
 *   → Still ONE database server
 *   → Still one logical table
 *   → Data split internally into partitions
 *   → Managed largely by the database engine
 *   → Goal: improve query EFFICIENCY and manageability
 *   → Complexity: relatively LOWER
 *
 * SHARDING:
 *   → Happens ACROSS multiple database instances
 *   → MULTIPLE database servers
 *   → Each shard has its own tables
 *   → Application usually needs routing logic
 *   → Goal: scale beyond a single machine
 *   → Complexity: MUCH HIGHER
 *
 * WHEN TO USE PARTITIONING INSTEAD OF SHARDING:
 *   → Data still fits on one server
 *   → Query performance is degrading due to table size
 *   → Many queries naturally filter by a dimension (date/region)
 *   → You need easier lifecycle management (archiving old data)
 *
 * WHEN TO USE SHARDING:
 *   → One machine is no longer enough
 *   → Storage ceiling is being reached
 *   → Write load cannot be handled by one database
 *   → Horizontal scale is necessary
 *
 * THE ESCALATION PATH:
 *   Partitioning often comes BEFORE sharding.
 *
 *   Optimize → Vertical Scale → Partitioning → Sharding
 */

In [ ]:
/**
 * Part-7: DATA LIFECYCLE
 */

In [ ]:
/**
 * Hot, Warm, aur Cold Data
 * ──────────────────────────
 *
 *   Saara data hamesha hot production DB mein nahi rehna chahiye.
 *   Purana data costly hai aur performance slow karta hai.
 *   Teen temperatures: Hot, Warm, Cold — har ek ke liye alag strategy.
 *
 * NOT ALL DATA SHOULD STAY IN HOT PRODUCTION DATABASE FOREVER.
 *
 * WHY DATA LIFECYCLE MATTERS:
 *   → Storage cost grows
 *   → Scans grow
 *   → Maintenance becomes harder
 *   → Performance suffers
 *   → Scaling pressure increases faster
 *
 * THE THREE TEMPERATURES:
 *
 *   1. HOT DATA
 *   ┌──────────────────────────────────────────────────────────┐
 *   │ • Recently and frequently queried                        │
 *   │ • Actively used in production flows                      │
 *   │ • Examples: last 30 days of orders, recent payments      │
 *   │ • Should stay in MAIN HOT PRODUCTION DATABASE            │
 *   └──────────────────────────────────────────────────────────┘
 *
 *   2. WARM DATA
 *   ┌──────────────────────────────────────────────────────────┐
 *   │ • Older data, still needed occasionally                  │
 *   │ • Not hit very often, but still useful                   │
 *   │ • Examples: 3-12 months of order history                 │
 *   │ • May stay close to operational system                   │
 *   └──────────────────────────────────────────────────────────┘
 *
 *   3. COLD DATA
 *   ┌──────────────────────────────────────────────────────────┐
 *   │ • Rarely queried                                         │
 *   │ • Mostly kept for audit, compliance, reporting           │
 *   │ • Examples: 2+ year old records, old logs                │
 *   │ • Should NOT live forever in hot production path         │
 *   └──────────────────────────────────────────────────────────┘
 *
 * HOW PLATFORMS HANDLE OLD DATA:
 *   1. KEEP RECENT DATA in main database (hot)
 *   2. ARCHIVE older data somewhere cheaper:
 *      → Archive tables
 *      → Data warehouse
 *      → Cheaper object storage
 *      → Historical storage systems
 *   3. DELETE or ANONYMIZE very old data
 *      → Based on retention policy
 *      → Based on privacy requirements
 *      → Based on compliance obligations
 */

In [ ]:
/**
 * Part-8: HOW PARTITIONING HELPS LIFECYCLE
 */

In [ ]:
/**
 * Partition Level Pe Operate Karo — Much Easier
 * ────────────────────────────────────────────────
 *
 *   Partitioning lifecycle management BAHUT EASIER banata hai.
 *   Row-by-row DELETE dangerous aur slow hai.
 *   Partition DROP karna instant aur safe hai.
*/

/**
 * WITHOUT PARTITIONING:
 *   Old data delete karne ke liye:
 */


DELETE FROM orders WHERE created_at < '2024-01-01';
-- DANGEROUS!
-- → Many row locks
-- → Large transactions
-- → Heavy undo/redo work
-- → More I/O
-- → Operational instability


/**
 * WITH PARTITIONING (by month):
 *   Old data delete karne ke liye:
 */


ALTER TABLE orders DROP PARTITION p2021_01;
ALTER TABLE orders DROP PARTITION p2021_02;
-- FAST AND CLEAN!
-- → Operates at storage level
-- → No row-by-row deletion
-- → Minimal I/O
-- → Instant (or near-instant) operation
-- → No transaction overhead


/**
 * WHY ROW-BY-ROW DELETE IS DANGEROUS:
 *   → Creates many row locks
 *   → Large transactions with long undo logs
 *   → Heavy redo/IO work
 *   → Can cause replication lag
 *   → May need to be batched over days/weeks
 *   → Risky for production
 *
 * PARTITION-BASED LIFECYCLE EXAMPLE:
 *   Keep last 6 months hot
 *   Archive older months
 *   Drop partitions older than 2 years
 */


-- Monthly archive job:
CREATE TABLE orders_archive_2025_01 LIKE orders;
INSERT INTO orders_archive_2025_01
SELECT * FROM orders
WHERE created_at BETWEEN '2025-01-01' AND '2025-01-31';

ALTER TABLE orders DROP PARTITION p2025_01;  -- Fast! Instant.

-- After retention period:
DROP TABLE orders_archive_2024_01;  -- Clean up


/**
 * This is WHY partitioning and lifecycle strategy are TIGHTLY CONNECTED.
 */

In [ ]:
/**
 * Part-9: DELETE VS ANONYMIZE
 */

In [ ]:
/**
 * Poora Hatana vs Identity Mask Karna
 * ──────────────────────────────────────
 *
 *   Delete = data permanently chala jaata hai.
 *   Anonymize = record rehta hai lekin PII (personal info) hata di jaati hai.
 *   Dono ka use case alag alag hai — legal/business needs pe depend karta hai.
 *
 * DELETE:
 *   → Data is permanently removed
 *   → Cannot be recovered
 *   → Used when data is no longer needed
 *   → Used when user requests deletion (GDPR right to erase)
 *
 * ANONYMIZE:
 *   → Record remains for business/audit purposes
 *   → Personal identifying fields are REMOVED or MASKED
 *   → Example: keep order record, but remove user_name, email, address
 *   → Useful when some record must stay but IDENTITY should not be associated
*/

/**
 * EXAMPLE: Anonymizing a user after account closure
 */

UPDATE orders
SET user_name       = '[REDACTED]',
    email           = NULL,
    address         = NULL,
    is_anonymized   = TRUE
WHERE user_id = 12345;


/**
 * WHY ANONYMIZE INSTEAD OF DELETE:
 *   → Legal/audit requirements to keep transaction records
 *   → Need to maintain business metrics without PII
 *   → Compliance with data retention laws
 *   → Ability to detect fraud without identifying individuals
 *
 * WHEN TO USE EACH:
 *   DELETE     → User requests full erasure, no legal retention need
 *   ANONYMIZE  → Data must be kept but identity must be removed
 */

In [ ]:
/**
 * Part-10: WHAT DECIDES LIFECYCLE POLICY
 */

In [ ]:
/**
 * Business, Legal, aur Operational Factors
 * ──────────────────────────────────────────
 *
 *   "X mahine ke baad delete karo" jaisi koi universal rule nahi hai.
 *   Lifecycle policy 5 factors pe depend karti hai.
 *   Alag industries ke alag rules hain.
 *
 * THERE IS NO UNIVERSAL RULE LIKE "DELETE AFTER X MONTHS."
 * Lifecycle policy depends on multiple factors:
 *
 *  1. BUSINESS VALUE
 *     → Is the data still useful to the product?
 *     → Does it power any features?
 *     → Is it needed for analytics or reporting?
 *     → What's the cost of keeping vs cost of retrieving?
 *
 *  2. ACCESS FREQUENCY
 *     → How often is this data queried?
 *     → Is it accessed daily, monthly, yearly, never?
 *     → Who accesses it? (users, support, analytics)
 *     → Can access patterns be changed?
 *
 *  3. LEGAL AND COMPLIANCE REQUIREMENTS
 *     → Do regulations require retention (e.g., 7 years)?
 *     → What data must be kept for audit?
 *     → What data must be deleted on request (GDPR, CCPA)?
 *     → What data can be anonymized vs must be deleted?
 *
 *  4. PRIVACY POLICY
 *     → Are you allowed to keep user data indefinitely?
 *     → What did you promise users in your privacy policy?
 *     → Do users have the right to request deletion?
 *     → Is consent required for continued storage?
 *
 *  5. STORAGE AND OPERATIONAL COST
 *     → How much does hot storage cost?
 *     → What's the cost of keeping vs archiving?
 *     → Does keeping data impact performance?
 *     → Is the cost worth the business value?
*/

/**
 * EXAMPLE POLICIES BY INDUSTRY:
 *
 *   E-commerce:
 *   → Hot: last 3 months orders
 *   → Warm: 3-12 months
 *   → Cold: archive after 1 year, delete after 7 years (tax compliance)
 *
 *   Healthcare:
 *   → Must retain 7-10 years (regulatory requirement)
 *   → Must anonymize after retention period
 *   → Strict deletion requirements
 *
 *   Social Media:
 *   → Hot: recent posts, comments
 *   → Delete on user request (GDPR)
 *   → Anonymize inactive accounts
 *
 *   Financial Services:
 *   → Must retain 5-7 years (regulatory requirement)
 *   → Cannot delete transactions
 *   → Can anonymize user identity
 */